# Gravity waves

In this section we will look at gravity waves. 

### Recap of derivation
We linearise our variables around $u = 0$, $v=0$ and $h=1$. Any second order terms are neglected. If we assume solutions of the form:
$$
(h, u, v ) = (\tilde{h}, \tilde{u}, \tilde{v})\exp\left(i(\omega t - k_x x - k_y y\right))
$$
this reduces to the matrix equation
$$
\begin{pmatrix}
i\omega & -ik_x & -ik_y \\
-ik_x & i\omega &-\text{Ro}^{-1} \\
-ik_y & \text{Ro}^{-1} & i\omega
\end{pmatrix}
\begin{pmatrix}
\tilde{h}\\ \tilde{u} \\ \tilde{v}
\end{pmatrix} = 0
$$

Non-zero solutions only occur when the determinant of the above matrix is non-zero. In this case:
$$
\omega^2 = k_x^2 + k_y^2 + \text{Ro}^{-2} = k^2 + \text{Ro}^{-2}.
$$
When rotation rate is low, solutions are non-dispersive with speed $c=1$. When rotation rate is high, waves become dispersive, i.e. the crests of the waves travel at a different speed to wave "packets", which travel at the group speed of the wave.

### For those using colab + google drive, run cells below

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

! pip install -e /content/drive/MyDrive/SW_summerschool/

### Now restart kernel using Runtime/Restart session in the colab menu

In [ ]:
import sys
import importlib
sys.modules["imp"] = importlib
from google.colab import output
output.enable_custom_widget_manager()

The config/output files for this exercise should be found/put in the directory `/content/drive/MyDrive/SW_summerschool/examples/Gravity_Waves`

In [ ]:
# Need to change into directory with config files
%cd /content/drive/MyDrive/SW_summerschool/examples/Gravity_Waves

### For those with local install, continue from below

In [ ]:
# Load the model again
%load_ext autoreload
%autoreload 2
%matplotlib widget
from IPython.display import HTML
import xarray as xr
import numpy as np
from sw_summerschool.plotting import animate_height_velocity, animate_contour
from sw_summerschool import SW_model
from sw_summerschool.helper import interp_to_centre
import matplotlib as mpl
import matplotlib.pyplot as plt


### The initial condition
To see the gravity waves well, we will initialise with a plane wave initial condition that is independent of the $y$ coordinate:

$$
\begin{align}
h_0 &= 1 + \delta \cos\left(\frac{2\pi x}{\lambda_x}\right) \\
u_0 &= \delta \cos\left(\frac{2\pi x}{\lambda_x}\right) \\
v_0 &= 0
\end{align}
$$

This should excite an exact solution to our wave equation.

**Task**:

Implement this in `initial_conditions.py`

In [ ]:
# Setup model with no rotation
model = SW_model("cfg_gravwaves.yaml", outfile = 'gravwave.nc', io_freq = 100, Ro=10000)

In [ ]:
model.integrate(10000)

In [ ]:
model.io.close()
ds = xr.load_dataset('gravwave.nc')
# Interpolate u and v to grid centres using helper function
u = interp_to_centre("u", ds, bc="periodic")
v = interp_to_centre("v", ds, bc="periodic")
q = interp_to_centre("q", ds, bc="periodic")

# Plotting functions take np arrays, so convert xarray dataarray into numpy array
h = ds["h"].data
t = ds["time"].data
x = ds["xmid"].data
y = ds["ymid"].data

# Use the plotting functions defined in plotting.py
from sw_summerschool.plotting import plot_height_velocity_snapshot, animate_height_velocity, plot_contour_snapshot
anim = animate_height_velocity(h, u, v, t, x, y, quiver_skip=2, quiver_scale=2, interval=100)

In [ ]:
plt.close('all')

## Further Tasks

Now that the initial conditions are sorted, run the model as in the Introduction and plot/animate the result.

A useful way to view the wave motion is to plot a **Hovmöller diagram**. This is a contour plot, where the $x$ axis is the dimension in which the wave is travelling (in our above case, the $x$ direction) and the $y$ axis is time. The "$z$" values of the contours will be the height, $h$. In a Hovmöller diagram, waves should appear as diagonal stripes, with the gradient $\Delta t/\Delta x$ being inversely proportional to the wavespeed.

1) Plot a Hovmöller diagram of your output, and use this to estimate the wavespeed in the model.
2) Try running the model with different values of the horizontal wavelength, $\lambda_x$. Does the wavespeed change? What does this tell us about the nature of the waves?
3) Turn on the effect of rotation by decreasing the Rossby number. What effect does this have on the wave motions/wave speeds?
4) A good way to see the effects of wave *dispersion* is to initialise a wave *packet*. Change the initial condition to:

$$
\begin{align}
h_0 &= 1 + \delta \cos\left(\frac{2\pi x}{\lambda_x}\right)\exp\left(-\frac{x^2}{2\sigma^2}\right) \\
u_0 &= \delta \cos\left(\frac{2\pi x}{\lambda_x}\right)\exp\left(-\frac{x^2}{2\sigma^2}\right) \\
v_0 &= 0
\end{align}
$$

i.e., where the waves are modulated by a Gaussian of width $\sigma$. To make the effects of dispersion pronounced, use $\sigma \gg \lambda_x$, i.e., the width of the modulating Gaussian profile should be much larger than the wavelength of your gravity waves. Describe the wave motions you see here.

In [ ]:
# Hovmöller diagram

plt.figure()
y_index = 15
plt.contourf(x, t, h[:,:,y_index])
plt.colorbar()
plt.xlabel('time')
plt.ylabel('x')

In [ ]:
# Setup model with rotation
model = SW_model("cfg_gravwaves.yaml", outfile = 'gravwave_Ro01.nc', io_freq = 100, Ro=0.1)
model.integrate(10000)

In [ ]:
model.io.close()
ds = xr.load_dataset('gravwave_Ro01.nc')
# Interpolate u and v to grid centres using helper function
u = interp_to_centre("u", ds, bc="periodic")
v = interp_to_centre("v", ds, bc="periodic")
q = interp_to_centre("q", ds, bc="periodic")

# Plotting functions take np arrays, so convert xarray dataarray into numpy array
h = ds["h"].data
t = ds["time"].data
x = ds["xmid"].data
y = ds["ymid"].data

# Use the plotting functions defined in plotting.py
from sw_summerschool.plotting import plot_height_velocity_snapshot, animate_height_velocity, plot_contour_snapshot
anim = animate_height_velocity(h, u, v, t, x, y, quiver_skip=2, quiver_scale=2, interval=100)

In [ ]:
plt.close('all')

In [ ]:
# Hovmöller diagram

plt.figure()
y_index = 15
plt.contourf(x, t, h[:,:,y_index])
plt.colorbar()
plt.xlabel('x')
plt.ylabel('t')
plt.ylim((0,2))
plt.show()

In [ ]:
# Setup model with no rotation
# Note for this model I uncommented
model = SW_model("cfg_gravwaves.yaml", outfile = 'gravwave_gaussian.nc', io_freq = 100, Ro=10000, print_freq=100)
model.integrate(10000)

In [ ]:
model.io.close()
ds = xr.load_dataset('gravwave_gaussian.nc')
# Interpolate u and v to grid centres using helper function
u = interp_to_centre("u", ds, bc="periodic")
v = interp_to_centre("v", ds, bc="periodic")
q = interp_to_centre("q", ds, bc="periodic")

# Plotting functions take np arrays, so convert xarray dataarray into numpy array
h = ds["h"].data
t = ds["time"].data
x = ds["xmid"].data
y = ds["ymid"].data

# Use the plotting functions defined in plotting.py
from sw_summerschool.plotting import plot_height_velocity_snapshot, animate_height_velocity, plot_contour_snapshot
anim = animate_height_velocity(h, u, v, t, x, y, quiver_skip=2, quiver_scale=2, interval=100)

In [ ]:
# Hovmöller diagram
plt.close('all')
plt.figure()
y_index = 15
plt.contourf(x, t, h[:,:,y_index])
plt.colorbar()
plt.xlabel('x')
plt.ylabel('t')
plt.ylim((0,2))
plt.show()

In [ ]:
# Setup model with no rotation
model = SW_model("cfg_gravwaves.yaml", outfile = 'gravwave_gaussian_rot.nc', io_freq = 100, Ro=0.1, print_freq=100)
model.integrate(10000)

In [ ]:
model.io.close()
ds = xr.load_dataset('gravwave_gaussian_rot.nc')
# Interpolate u and v to grid centres using helper function
u = interp_to_centre("u", ds, bc="periodic")
v = interp_to_centre("v", ds, bc="periodic")
q = interp_to_centre("q", ds, bc="periodic")

# Plotting functions take np arrays, so convert xarray dataarray into numpy array
h = ds["h"].data
t = ds["time"].data
x = ds["xmid"].data
y = ds["ymid"].data

# Use the plotting functions defined in plotting.py
from sw_summerschool.plotting import plot_height_velocity_snapshot, animate_height_velocity, plot_contour_snapshot
anim = animate_height_velocity(h, u, v, t, x, y, quiver_skip=2, quiver_scale=2, interval=100)

In [ ]:
# Hovmöller diagram
plt.close('all')
plt.figure()
y_index = 15
plt.contourf(x, t, h[:,:,y_index])
plt.colorbar()
plt.xlabel('x')
plt.ylabel('t')
plt.ylim((0,2))
plt.show()